# Lab 20: Statistical Output Analysis and Precision Estimation

**Problem Statement:**
This lab problem evaluates customer flow at an automated drive-thru pharmacy kiosk operating as a terminating simulation system until 50 cars are served. Students are provided with average customer waiting times across five independent replication runs (3.2, 4.3, 5.1, 4.2, and 4.6 minutes).

**Tasks:**
1. Calculate overall point estimate (sample mean).
2. Compute sample variance and standard error.
3. Construct a 95% confidence interval using Student's t-distribution.
4. Determine total number of simulation replications required to achieve a desired half-width precision of no more than 0.5 minutes.


### Theory and Formulas

Given $n$ independent replications $X_1, X_2, \dots, X_n$:
1. **Point Estimate (Sample Mean)**: $\\bar{X} = \\frac{1}{n}\sum X_i$
2. **Sample Variance**: $S^2 = \\frac{1}{n-1}\sum (X_i - \\bar{X})^2$
3. **Standard Error**: $SE = \\frac{S}{\\sqrt{n}}$
4. **95% Confidence Interval**: $\\bar{X} \\pm h$, where half-width $h = t_{n-1, 0.025} \\times SE$
5. **Required Replications for precision $\epsilon$**:
   An approximate formula to find the required number of replications $n^*$ to achieve a half-width of at most $\epsilon$ is:
   $$n^* \\approx n \\left( \\frac{h}{\epsilon} \\right)^2$$
   *(Alternatively, iterating $n^*$ until $t_{n^*-1, 0.025} \\times \\frac{S}{\\sqrt{n^*}} \le \epsilon$)*


In [ ]:
import numpy as np
import scipy.stats as stats
import math

# Data provided
replications = [3.2, 4.3, 5.1, 4.2, 4.6]
n = len(replications)
desired_precision = 0.5

print("--- Output Analysis for Terminating Simulation ---")
print(f"Data from {n} runs: {replications}")

# 1. Point Estimate (Mean)
sample_mean = np.mean(replications)
print(f"\n1. Point Estimate (Sample Mean): {sample_mean:.3f} mins")

# 2. Sample Variance and Standard Error
# ddof=1 ensures we divide by n-1 for sample variance
sample_var = np.var(replications, ddof=1)
sample_std = np.std(replications, ddof=1)
std_error = sample_std / np.sqrt(n)

print(f"\n2. Sample Variance (S²)      : {sample_var:.3f}")
print(f"   Sample Std Dev (S)        : {sample_std:.3f}")
print(f"   Standard Error (SE)       : {std_error:.3f}")

# 3. 95% Confidence Interval
alpha = 0.05
# Two-tailed t-value for 95% CI and n-1 degrees of freedom
t_crit = stats.t.ppf(1 - alpha/2, df=n-1)

half_width = t_crit * std_error
ci_lower = sample_mean - half_width
ci_upper = sample_mean + half_width

print(f"\n3. 95% Confidence Interval   : [{ci_lower:.3f}, {ci_upper:.3f}]")
print(f"   t-critical value (df={n-1}) : {t_crit:.3f}")
print(f"   Half-width (h)            : {half_width:.3f} mins")

# 4. Required Replications
# Approximate method: n* = n * (h / epsilon)^2
n_star_approx = int(math.ceil(n * (half_width / desired_precision)**2))

# Iterative exact method
n_star_exact = n
while True:
    t_val = stats.t.ppf(1 - alpha/2, df=n_star_exact-1)
    hw = t_val * (sample_std / np.sqrt(n_star_exact))
    if hw <= desired_precision:
        break
    n_star_exact += 1

print(f"\n4. Required Replications for precision ε ≤ {desired_precision}:")
print(f"   Using Approximation Formula : {n_star_approx} total replications")
print(f"   Using Iterative Search      : {n_star_exact} total replications")
print(f"   (We need to run {n_star_exact - n} more replications)")
